## Imports and data loading

In [70]:
import pandas as pd 
import numpy as np
import warnings
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from collections import defaultdict
warnings.filterwarnings('ignore')

In [71]:
df_train = pd.read_json('data/train.json')
df_test = pd.read_json('data/test.json')

## Data Analysis

In [72]:
df_train.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


In [73]:
low_border = df_train['price'].quantile(0.01)
high_border = df_train['price'].quantile(0.99)

df_train = df_train[(df_train['price'] >= low_border) & (df_train['price'] <= high_border)]

In [74]:
print(df_train['features'].head())

4     [Dining Room, Pre-War, Laundry in Building, Di...
6     [Doorman, Elevator, Laundry in Building, Dishw...
9     [Doorman, Elevator, Laundry in Building, Laund...
10                                                   []
15    [Doorman, Elevator, Fitness Center, Laundry in...
Name: features, dtype: object


In [75]:
all_features = []
for _, row in df_train.iterrows():
    cleaned = str(row['features'])
    for ch in ['[', ']', "'", '"']:
        cleaned = cleaned.replace(ch, '')
    values = [v.strip() for v in cleaned.split(',') if v.strip()]
    all_features.extend(values)
print(f"Уникальных значений: {len(set(all_features))}")

Уникальных значений: 1537


In [76]:
from collections import Counter

counter = Counter(all_features)
features = sorted(counter.items(), key=lambda x: -x[1])
top_20 = features[:20]
top_20

[('Elevator', 25398),
 ('Hardwood Floors', 23159),
 ('Cats Allowed', 23148),
 ('Dogs Allowed', 21662),
 ('Doorman', 20497),
 ('Dishwasher', 20095),
 ('No Fee', 17806),
 ('Laundry in Building', 16093),
 ('Fitness Center', 13000),
 ('Pre-War', 8978),
 ('Laundry in Unit', 8448),
 ('Roof Deck', 6423),
 ('Outdoor Space', 5137),
 ('Dining Room', 4901),
 ('High Speed Internet', 4225),
 ('Balcony', 2898),
 ('Swimming Pool', 2648),
 ('Laundry In Building', 2565),
 ('New Construction', 2507),
 ('Terrace', 2179)]

In [77]:
top_20_names = [name for name, count in top_20]

In [78]:
for feature in top_20_names:
    df_train[feature] = df_train['features'].apply(
        lambda x: 1 if feature in str(x) else 0
    )
feature_list = top_20_names + ['bathrooms', 'bedrooms']
print(f"Всего признаков: {len(feature_list)}")
feature_list

Всего признаков: 22


['Elevator',
 'Hardwood Floors',
 'Cats Allowed',
 'Dogs Allowed',
 'Doorman',
 'Dishwasher',
 'No Fee',
 'Laundry in Building',
 'Fitness Center',
 'Pre-War',
 'Laundry in Unit',
 'Roof Deck',
 'Outdoor Space',
 'Dining Room',
 'High Speed Internet',
 'Balcony',
 'Swimming Pool',
 'Laundry In Building',
 'New Construction',
 'Terrace',
 'bathrooms',
 'bedrooms']

In [79]:
from sklearn.model_selection import train_test_split

X = df_train[feature_list].values
y = df_train['price'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (38703, 22), Test: (9676, 22)


## Models implementation - Linear regression

In [80]:
class LinearRegressionAnalytical:
        
    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        return self
    
    def predict(self, X):
        X_b = np.c_[np.ones(len(X)), X]
        return X_b @ self.w

In [81]:
class LinearRegressionGD:
    def __init__(self, lr=0.01, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter
    
    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)
        for _ in range(self.n_iter):
            grad = (2/n) * X_b.T @ (X_b @ self.w - y)
            self.w -= self.lr * grad
        return self
    
    def predict(self, X):
        X_b = np.c_[np.ones(len(X)), X]
        return X_b @ self.w


In [82]:
class LinearRegressionSGD:
    def __init__(self, lr=0.01, n_iter=1000, random_state=42):
        self.lr = lr
        self.n_iter = n_iter
        self.random_state = random_state
    
    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)
        for _ in range(self.n_iter):
            idx = rng.randint(0, n)
            xi = X_b[idx:idx+1]
            yi = y[idx:idx+1]
            grad = 2 * xi.T @ (xi @ self.w - yi)
            self.w -= self.lr * grad
        return self
    
    def predict(self, X):
        X_b = np.c_[np.ones(len(X)), X]
        return X_b @ self.w


In [83]:
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

In [84]:
from sklearn.linear_model import LinearRegression as SklearnLR

models = {
    'Analytical': LinearRegressionAnalytical(),
    'GD':         LinearRegressionGD(lr=0.01, n_iter=2000),
    'SGD':        LinearRegressionSGD(lr=0.01, n_iter=10000, random_state=42),
    'Sklearn':    SklearnLR(),
}

results_mae  = []
results_rmse = []
results_r2   = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    pred_train = model.predict(X_train)
    pred_test  = model.predict(X_test)
    
    results_mae.append({
        'model': name,
        'train': mae(y_train, pred_train),
        'test':  mae(y_test,  pred_test),
    })
    results_rmse.append({
        'model': name,
        'train': rmse(y_train, pred_train),
        'test':  rmse(y_test,  pred_test),
    })
    results_r2.append({
        'model': name,
        'train': r2_score(y_train, pred_train),
        'test':  r2_score(y_test,  pred_test),
    })


In [85]:
print("MAE:")
print(pd.DataFrame(results_mae).set_index('model').round(2))

print("\nRMSE:")
print(pd.DataFrame(results_rmse).set_index('model').round(2))

print("\nR2:")
print(pd.DataFrame(results_r2).set_index('model').round(4))

MAE:
             train    test
model                     
Analytical  710.99  713.65
GD          710.96  713.53
SGD         808.43  820.82
Sklearn     710.99  713.65

RMSE:
              train     test
model                       
Analytical  1033.84  1041.84
GD          1034.04  1042.02
SGD         1213.64  1224.07
Sklearn     1033.84  1041.84

R2:
             train    test
model                     
Analytical  0.5821  0.5713
GD          0.5819  0.5712
SGD         0.4241  0.4083
Sklearn     0.5821  0.5713


## Regularized models implementation — Ridge, Lasso, ElasticNet

In [86]:
class RidgeSGD:    
    def __init__(self, lr=0.01, n_iter=1000, alpha=1.0, random_state=42):
        self.lr = lr
        self.n_iter = n_iter
        self.alpha = alpha
        self.random_state = random_state
    
    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)
        
        for _ in range(self.n_iter):
            idx = rng.randint(0, n)
            xi = X_b[idx:idx+1]
            yi = y[idx:idx+1]
            grad = 2 * xi.T @ (xi @ self.w - yi)
            grad[1:] += 2 * self.alpha * self.w[1:]
            self.w -= self.lr * grad
        return self
    
    def predict(self, X):
        X_b = np.c_[np.ones(len(X)), X]
        return X_b @ self.w


class LassoSGD:    
    def __init__(self, lr=0.01, n_iter=1000, alpha=1.0, random_state=42):
        self.lr = lr
        self.n_iter = n_iter
        self.alpha = alpha
        self.random_state = random_state
    
    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)
        
        for _ in range(self.n_iter):
            idx = rng.randint(0, n)
            xi = X_b[idx:idx+1]
            yi = y[idx:idx+1]
            grad = 2 * xi.T @ (xi @ self.w - yi)
            grad[1:] += self.alpha * np.sign(self.w[1:])
            self.w -= self.lr * grad
        return self
    
    def predict(self, X):
        X_b = np.c_[np.ones(len(X)), X]
        return X_b @ self.w


class ElasticNetSGD:    
    def __init__(self, lr=0.01, n_iter=1000, alpha=1.0, l1_ratio=0.5, random_state=42):
        self.lr = lr
        self.n_iter = n_iter
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.random_state = random_state
    
    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)
        
        for _ in range(self.n_iter):
            idx = rng.randint(0, n)
            xi = X_b[idx:idx+1]
            yi = y[idx:idx+1]
            grad = 2 * xi.T @ (xi @ self.w - yi)
            grad[1:] += self.alpha * self.l1_ratio * np.sign(self.w[1:])
            grad[1:] += self.alpha * (1 - self.l1_ratio) * 2 * self.w[1:]
            self.w -= self.lr * grad
        return self
    
    def predict(self, X):
        X_b = np.c_[np.ones(len(X)), X]
        return X_b @ self.w

In [87]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

models_reg = {
    'Ridge (our)':       RidgeSGD(lr=0.01, n_iter=10000, alpha=1.0, random_state=42),
    'Lasso (our)':       LassoSGD(lr=0.01, n_iter=10000, alpha=1.0, random_state=42),
    'ElasticNet (our)':  ElasticNetSGD(lr=0.01, n_iter=10000, alpha=1.0, l1_ratio=0.5, random_state=42),
    'Ridge (sklearn)':   Ridge(alpha=1.0),
    'Lasso (sklearn)':   Lasso(alpha=1.0),
    'ElasticNet (sklearn)': ElasticNet(alpha=1.0, l1_ratio=0.5),
}

results_mae  = []
results_rmse = []
results_r2   = []

for name, model in models_reg.items():
    model.fit(X_train, y_train)
    
    pred_train = model.predict(X_train)
    pred_test  = model.predict(X_test)
    
    results_mae.append({
        'model': name,
        'train': mae(y_train, pred_train),
        'test':  mae(y_test,  pred_test),
    })
    results_rmse.append({
        'model': name,
        'train': rmse(y_train, pred_train),
        'test':  rmse(y_test,  pred_test),
    })
    results_r2.append({
        'model': name,
        'train': r2_score(y_train, pred_train),
        'test':  r2_score(y_test,  pred_test),
    })

In [88]:
print("MAE:")
print(pd.DataFrame(results_mae).set_index('model').round(2))

print("\nRMSE:")
print(pd.DataFrame(results_rmse).set_index('model').round(2))

print("\nR2:")
print(pd.DataFrame(results_r2).set_index('model').round(4))

MAE:
                       train    test
model                               
Ridge (our)           894.19  903.29
Lasso (our)           807.49  819.82
ElasticNet (our)      838.42  847.42
Ridge (sklearn)       710.99  713.64
Lasso (sklearn)       710.54  713.34
ElasticNet (sklearn)  805.45  809.96

RMSE:
                        train     test
model                                 
Ridge (our)           1425.65  1429.08
Lasso (our)           1213.13  1223.46
ElasticNet (our)      1351.00  1356.20
Ridge (sklearn)       1033.84  1041.84
Lasso (sklearn)       1034.03  1042.02
ElasticNet (sklearn)  1189.19  1190.42

R2:
                       train    test
model                               
Ridge (our)           0.2053  0.1935
Lasso (our)           0.4246  0.4089
ElasticNet (our)      0.2863  0.2736
Ridge (sklearn)       0.5821  0.5714
Lasso (sklearn)       0.5819  0.5712
ElasticNet (sklearn)  0.4471  0.4404


## Feature normalization

In [89]:
# Нормализация ОБЯЗАТЕЛЬНА:
# 1. Градиентный спуск — признаки с большим масштабом доминируют,
#    градиент будет делать огромные шаги по одной оси и маленькие по другой
# 2. KNN, K-means, SVM — расстояния между точками теряют смысл если признаки в разных масштабах
#    (площадь 50м² vs цена 5_000_000 — цена полностью подавит площадь)
# 3. Регуляризация (Ridge, Lasso) — штраф применяется к весам,
#    но веса зависят от масштаба признаков — несправедливый штраф
# 4. PCA, нейронные сети

# Нормализация НЕ НУЖНА:
# 1. Decision Tree, Random Forest — деревья делят по порогам,
#    масштаб признака не влияет на качество разбиения
# 2. Когда все признаки уже в одном масштабе (например бинарные 0/1)

In [90]:
class MinMaxScalerCustom:
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
    
    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_)
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [91]:
from sklearn.preprocessing import MinMaxScaler

scaler_custom = MinMaxScalerCustom()
X_train_mm_custom = scaler_custom.fit_transform(X_train)
X_test_mm_custom  = scaler_custom.transform(X_test)

scaler_sklearn = MinMaxScaler()
X_train_mm_sklearn = scaler_sklearn.fit_transform(X_train)
X_test_mm_sklearn  = scaler_sklearn.transform(X_test)

print("Максимальная разница (train):",
      np.abs(X_train_mm_custom - X_train_mm_sklearn).max())
print("Максимальная разница (test):",
      np.abs(X_test_mm_custom - X_test_mm_sklearn).max())

Максимальная разница (train): 5.551115123125783e-17
Максимальная разница (test): 5.551115123125783e-17


In [92]:
class StandardScalerCustom:
    def fit(self, X):
        self.mean_ = X.mean(axis=0)   
        self.std_  = X.std(axis=0)    
        return self
    
    def transform(self, X):
        return (X - self.mean_) / self.std_
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [93]:
from sklearn.preprocessing import StandardScaler

scaler_std_custom = StandardScalerCustom()
X_train_std_custom = scaler_std_custom.fit_transform(X_train)
X_test_std_custom  = scaler_std_custom.transform(X_test)

scaler_std_sklearn = StandardScaler()
X_train_std_sklearn = scaler_std_sklearn.fit_transform(X_train)
X_test_std_sklearn  = scaler_std_sklearn.transform(X_test)

print("Максимальная разница (train):",
      np.abs(X_train_std_custom - X_train_std_sklearn).max())
print("Максимальная разница (test):",
      np.abs(X_test_std_custom - X_test_std_sklearn).max())

Максимальная разница (train): 0.0
Максимальная разница (test): 0.0


## Fit custom and sklearn models with normalized data

In [94]:
def get_models():
    return {
        'LinearRegression (our)':    LinearRegressionSGD(lr=0.01, n_iter=10000, random_state=42),
        'Ridge (our)':               RidgeSGD(lr=0.01, n_iter=10000, alpha=1.0, random_state=42),
        'Lasso (our)':               LassoSGD(lr=0.01, n_iter=10000, alpha=1.0, random_state=42),
        'ElasticNet (our)':          ElasticNetSGD(lr=0.01, n_iter=10000, alpha=1.0, random_state=42),
        'LinearRegression (sklearn)': SklearnLR(),
        'Ridge (sklearn)':            Ridge(alpha=1.0),
        'Lasso (sklearn)':            Lasso(alpha=1.0),
        'ElasticNet (sklearn)':       ElasticNet(alpha=1.0, l1_ratio=0.5),
    }

def fit_and_evaluate(models, X_tr, X_te, y_tr, y_te, prefix=''):
    rows_mae, rows_rmse, rows_r2 = [], [], []
    
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        pred_train = model.predict(X_tr)
        pred_test  = model.predict(X_te)
        
        label = f"{prefix}{name}" if prefix else name
        
        rows_mae.append({
            'model': label,
            'train': mae(y_tr, pred_train),
            'test':  mae(y_te, pred_test),
        })
        rows_rmse.append({
            'model': label,
            'train': rmse(y_tr, pred_train),
            'test':  rmse(y_te, pred_test),
        })
        rows_r2.append({
            'model': label,
            'train': r2_score(y_tr, pred_train),
            'test':  r2_score(y_te, pred_test),
        })
    
    return rows_mae, rows_rmse, rows_r2

mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train)
X_test_mm  = mm_scaler.transform(X_test)

mae_mm, rmse_mm, r2_mm = fit_and_evaluate(
    get_models(), X_train_mm, X_test_mm, y_train, y_test,
    prefix='[MinMax] '
)

std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std  = std_scaler.transform(X_test)

mae_std, rmse_std, r2_std = fit_and_evaluate(
    get_models(), X_train_std, X_test_std, y_train, y_test,
    prefix='[Standard] '
)

all_mae  = results_mae  + mae_mm  + mae_std
all_rmse = results_rmse + rmse_mm + rmse_std
all_r2   = results_r2   + r2_mm   + r2_std

df_mae  = pd.DataFrame(all_mae).set_index('model').round(2)
df_rmse = pd.DataFrame(all_rmse).set_index('model').round(2)
df_r2   = pd.DataFrame(all_r2).set_index('model').round(4)

print("MAE:")
print(df_mae)

print("\nRMSE:")
print(df_rmse)

print("\nR2:")
print(df_r2)

MAE:
                                         train     test
model                                                  
Ridge (our)                             894.19   903.29
Lasso (our)                             807.49   819.82
ElasticNet (our)                        838.42   847.42
Ridge (sklearn)                         710.99   713.64
Lasso (sklearn)                         710.54   713.34
ElasticNet (sklearn)                    805.45   809.96
[MinMax] LinearRegression (our)         763.81   763.11
[MinMax] Ridge (our)                   1044.96  1054.87
[MinMax] Lasso (our)                    763.33   762.65
[MinMax] ElasticNet (our)              1018.34  1029.97
[MinMax] LinearRegression (sklearn)     710.99   713.65
[MinMax] Ridge (sklearn)                711.09   713.59
[MinMax] Lasso (sklearn)                710.82   713.32
[MinMax] ElasticNet (sklearn)          1054.84  1064.27
[Standard] LinearRegression (our)       787.71   793.64
[Standard] Ridge (our)                  869

## Overfit models

In [95]:
X_base = df_train[['bathrooms', 'bedrooms']].values
y = df_train['price'].values

X_base_train, X_base_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

poly = PolynomialFeatures(degree=10)
X_poly_train = poly.fit_transform(X_base_train)
X_poly_test  = poly.transform(X_base_test)

In [96]:
std_scaler_poly = StandardScaler()
X_poly_train_scaled = std_scaler_poly.fit_transform(X_poly_train)
X_poly_test_scaled  = std_scaler_poly.transform(X_poly_test)

In [97]:
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

alpha_results_mae  = []
alpha_results_rmse = []
alpha_results_r2   = []

for alpha in alphas:
    models_poly = {
        f'LinearRegression (poly)':          LinearRegressionSGD(lr=0.001, n_iter=10000, random_state=42),
        f'Ridge alpha={alpha} (our)':        RidgeSGD(lr=0.001, n_iter=10000, alpha=alpha, random_state=42),
        f'Lasso alpha={alpha} (our)':        LassoSGD(lr=0.001, n_iter=10000, alpha=alpha, random_state=42),
        f'ElasticNet alpha={alpha} (our)':   ElasticNetSGD(lr=0.001, n_iter=10000, alpha=alpha, random_state=42),
        f'Ridge alpha={alpha} (sklearn)':    Ridge(alpha=alpha),
        f'Lasso alpha={alpha} (sklearn)':    Lasso(alpha=alpha),
        f'ElasticNet alpha={alpha} (sklearn)': ElasticNet(alpha=alpha, l1_ratio=0.5),
    }
    
    mae_p, rmse_p, r2_p = fit_and_evaluate(
        models_poly,
        X_poly_train_scaled, X_poly_test_scaled,
        y_train, y_test,
        prefix=''
    )
    
    alpha_results_mae.extend(mae_p)
    alpha_results_rmse.extend(rmse_p)
    alpha_results_r2.extend(r2_p)

In [98]:
df_poly_mae  = pd.DataFrame(alpha_results_mae).set_index('model').round(2)
df_poly_rmse = pd.DataFrame(alpha_results_rmse).set_index('model').round(2)
df_poly_r2   = pd.DataFrame(alpha_results_r2).set_index('model').round(4)

print("MAE:")
print(df_poly_mae)
print("\nRMSE:")
print(df_poly_rmse)
print("\nR2:")
print(df_poly_r2)

best_model = df_poly_r2['test'].idxmax()
print(f"\nЛучшая модель: {best_model}")
print(df_poly_r2.loc[best_model])

MAE:
                                         train          test
model                                                       
LinearRegression (poly)           3.655355e+14  3.580867e+14
Ridge alpha=0.001 (our)           3.599383e+14  3.526042e+14
Lasso alpha=0.001 (our)           3.655457e+14  3.580967e+14
ElasticNet alpha=0.001 (our)      3.627316e+14  3.553403e+14
Ridge alpha=0.001 (sklearn)       7.563400e+02  7.597900e+02
Lasso alpha=0.001 (sklearn)       7.613900e+02  7.640100e+02
ElasticNet alpha=0.001 (sklearn)  7.617200e+02  7.644300e+02
LinearRegression (poly)           3.655355e+14  3.580867e+14
Ridge alpha=0.01 (our)            3.131173e+14  3.067428e+14
Lasso alpha=0.01 (our)            3.656378e+14  3.581870e+14
ElasticNet alpha=0.01 (our)       3.384037e+14  3.315111e+14
Ridge alpha=0.01 (sklearn)        7.567600e+02  7.600100e+02
Lasso alpha=0.01 (sklearn)        7.614000e+02  7.640200e+02
ElasticNet alpha=0.01 (sklearn)   7.656300e+02  7.684000e+02
LinearRegression (p

## Native models

In [99]:
from sklearn.metrics import mean_squared_error

mean_value   = y_train.mean()
median_value = np.median(y_train)

naive_models = {
    'naive_mean':   mean_value,
    'naive_median': median_value,
}

for name, value in naive_models.items():
    y_train_pred = np.full(len(y_train), value)
    y_test_pred  = np.full(len(y_test),  value)

    all_mae.append({
        'model': name,
        'train': mae(y_train, y_train_pred),
        'test':  mae(y_test,  y_test_pred),
    })
    all_rmse.append({
        'model': name,
        'train': rmse(y_train, y_train_pred),
        'test':  rmse(y_test,  y_test_pred),
    })
    all_r2.append({
        'model': name,
        'train': r2_score(y_train, y_train_pred),
        'test':  r2_score(y_test,  y_test_pred),
    })

df_mae  = pd.DataFrame(all_mae).set_index('model').round(2)
df_rmse = pd.DataFrame(all_rmse).set_index('model').round(2)
df_r2   = pd.DataFrame(all_r2).set_index('model').round(4)

print("MAE:")
print(df_mae)
print("\nRMSE:")
print(df_rmse)
print("\nR2:")
print(df_r2)

MAE:
                                         train     test
model                                                  
Ridge (our)                             894.19   903.29
Lasso (our)                             807.49   819.82
ElasticNet (our)                        838.42   847.42
Ridge (sklearn)                         710.99   713.64
Lasso (sklearn)                         710.54   713.34
ElasticNet (sklearn)                    805.45   809.96
[MinMax] LinearRegression (our)         763.81   763.11
[MinMax] Ridge (our)                   1044.96  1054.87
[MinMax] Lasso (our)                    763.33   762.65
[MinMax] ElasticNet (our)              1018.34  1029.97
[MinMax] LinearRegression (sklearn)     710.99   713.65
[MinMax] Ridge (sklearn)                711.09   713.59
[MinMax] Lasso (sklearn)                710.82   713.32
[MinMax] ElasticNet (sklearn)          1054.84  1064.27
[Standard] LinearRegression (our)       787.71   793.64
[Standard] Ridge (our)                  869

## Compare results

In [100]:
print("=" * 60)
print("MAE (меньше = лучше)")
print("=" * 60)
print(df_mae.sort_values('test'))

print("\n" + "=" * 60)
print("RMSE (меньше = лучше)")
print("=" * 60)
print(df_rmse.sort_values('test'))

print("\n" + "=" * 60)
print("R2 (больше = лучше)")
print("=" * 60)
print(df_r2.sort_values('test', ascending=False))

best_mae  = df_mae['test'].idxmin()
best_rmse = df_rmse['test'].idxmin()
best_r2   = df_r2['test'].idxmax()

print("\n" + "=" * 60)
print("Лучшие модели по метрикам:")
print("=" * 60)
print(f"По MAE:  {best_mae}")
print(f"  train={df_mae.loc[best_mae, 'train']:.2f}, test={df_mae.loc[best_mae, 'test']:.2f}")
print(f"По RMSE: {best_rmse}")
print(f"  train={df_rmse.loc[best_rmse, 'train']:.2f}, test={df_rmse.loc[best_rmse, 'test']:.2f}")
print(f"По R2:   {best_r2}")
print(f"  train={df_r2.loc[best_r2, 'train']:.4f}, test={df_r2.loc[best_r2, 'test']:.4f}")

df_r2['gap'] = (df_r2['train'] - df_r2['test']).abs()

good_models = df_r2[df_r2['test'] > 0.3]
most_stable = good_models['gap'].idxmin()

print("\n" + "=" * 60)
print("Самая стабильная модель (минимальный разрыв train/test):")
print("=" * 60)
print(f"Модель: {most_stable}")
print(good_models.loc[most_stable])

print("\nТоп-5 стабильных моделей:")
print(good_models.sort_values('gap').head())

MAE (меньше = лучше)
                                         train     test
model                                                  
[MinMax] Lasso (sklearn)                710.82   713.32
Lasso (sklearn)                         710.54   713.34
[Standard] Lasso (sklearn)              710.78   713.50
[MinMax] Ridge (sklearn)                711.09   713.59
Ridge (sklearn)                         710.99   713.64
[MinMax] LinearRegression (sklearn)     710.99   713.65
[Standard] Ridge (sklearn)              710.99   713.65
[Standard] LinearRegression (sklearn)   710.99   713.65
[Standard] ElasticNet (sklearn)         740.71   745.13
[MinMax] Lasso (our)                    763.33   762.65
[MinMax] LinearRegression (our)         763.81   763.11
[Standard] Lasso (our)                  787.22   793.18
[Standard] LinearRegression (our)       787.71   793.64
ElasticNet (sklearn)                    805.45   809.96
Lasso (our)                             807.49   819.82
[Standard] ElasticNet (our)

## *Additional task*

In [101]:
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

model_log = Ridge(alpha=0.001)
model_log.fit(X_train_std, y_train_log)

pred_train_log = np.expm1(model_log.predict(X_train_std))  # exp(y) - 1
pred_test_log  = np.expm1(model_log.predict(X_test_std))

print("Ridge + log(target):")
print(f"MAE  train={mae(y_train, pred_train_log):.2f}, test={mae(y_test, pred_test_log):.2f}")
print(f"RMSE train={rmse(y_train, pred_train_log):.2f}, test={rmse(y_test, pred_test_log):.2f}")
print(f"R2   train={r2_score(y_train, pred_train_log):.4f}, test={r2_score(y_test, pred_test_log):.4f}")

all_mae.append({'model': 'Ridge + log(y)', 'train': mae(y_train, pred_train_log), 'test': mae(y_test, pred_test_log)})
all_rmse.append({'model': 'Ridge + log(y)', 'train': rmse(y_train, pred_train_log), 'test': rmse(y_test, pred_test_log)})
all_r2.append({'model': 'Ridge + log(y)', 'train': r2_score(y_train, pred_train_log), 'test': r2_score(y_test, pred_test_log)})

Ridge + log(target):
MAE  train=696.84, test=702.84
RMSE train=1062.33, test=1049.24
R2   train=0.5587, test=0.5652


In [102]:
Q1 = np.percentile(y_train, 25)
Q3 = np.percentile(y_train, 75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

mask = (y_train >= lower) & (y_train <= upper)

X_train_clean = X_train_std[mask]
y_train_clean = y_train[mask]

print(f"Удалено выбросов: {(~mask).sum()} из {len(y_train)}")
print(f"Осталось: {mask.sum()} примеров")

model_clean = Ridge(alpha=0.001)
model_clean.fit(X_train_clean, y_train_clean)

pred_train_clean = model_clean.predict(X_train_std)   # оцениваем на полном трейне
pred_test_clean  = model_clean.predict(X_test_std)    # и на полном тесте

print("\nRidge без выбросов в трейне:")
print(f"MAE  train={mae(y_train, pred_train_clean):.2f}, test={mae(y_test, pred_test_clean):.2f}")
print(f"RMSE train={rmse(y_train, pred_train_clean):.2f}, test={rmse(y_test, pred_test_clean):.2f}")
print(f"R2   train={r2_score(y_train, pred_train_clean):.4f}, test={r2_score(y_test, pred_test_clean):.4f}")

all_mae.append({'model': 'Ridge no outliers', 'train': mae(y_train, pred_train_clean), 'test': mae(y_test, pred_test_clean)})
all_rmse.append({'model': 'Ridge no outliers', 'train': rmse(y_train, pred_train_clean), 'test': rmse(y_test, pred_test_clean)})
all_r2.append({'model': 'Ridge no outliers', 'train': r2_score(y_train, pred_train_clean), 'test': r2_score(y_test, pred_test_clean)})

Удалено выбросов: 2158 из 38703
Осталось: 36545 примеров

Ridge без выбросов в трейне:
MAE  train=724.10, test=728.69
RMSE train=1120.70, test=1123.28
R2   train=0.5089, test=0.5017


In [103]:
class LinearRegressionBatchGD:
    def __init__(self, lr=0.01, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter

    def fit(self, X, y):
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)

        for _ in range(self.n_iter):
            grad = (2/n) * X_b.T @ (X_b @ self.w - y)
            self.w -= self.lr * grad
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w


class LinearRegressionMiniBatchGD:
    def __init__(self, lr=0.01, n_iter=1000, batch_size=32, random_state=42):
        self.lr = lr
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        X_b = np.c_[np.ones(len(X)), X]
        self.w = np.zeros(X_b.shape[1])
        n = len(y)

        for _ in range(self.n_iter):
            idx = rng.choice(n, self.batch_size, replace=False)
            xi = X_b[idx]
            yi = y[idx]
            grad = (2/self.batch_size) * xi.T @ (xi @ self.w - yi)
            self.w -= self.lr * grad
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w


gd_models = {
    'SGD':            LinearRegressionSGD(lr=0.01, n_iter=10000, random_state=42),
    'Batch GD':       LinearRegressionBatchGD(lr=0.01, n_iter=1000),
    'Mini-batch GD':  LinearRegressionMiniBatchGD(lr=0.01, n_iter=1000, batch_size=32),
}

print(f"{'Метод':<20} {'MAE train':>12} {'MAE test':>12} {'R2 train':>10} {'R2 test':>10}")
print("-" * 65)
for name, model in gd_models.items():
    model.fit(X_train_std, y_train)
    p_tr = model.predict(X_train_std)
    p_te = model.predict(X_test_std)
    print(f"{name:<20} {mae(y_train,p_tr):>12.2f} {mae(y_test,p_te):>12.2f} "
          f"{r2_score(y_train,p_tr):>10.4f} {r2_score(y_test,p_te):>10.4f}")

Метод                   MAE train     MAE test   R2 train    R2 test
-----------------------------------------------------------------
SGD                        787.71       793.64     0.5006     0.4913
Batch GD                   711.01       713.65     0.5821     0.5714
Mini-batch GD              710.47       712.83     0.5771     0.5684
